<a href="https://colab.research.google.com/github/Nakib-Nasrullah/Dengue_Research/blob/main/Dengue_2nd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Data handling and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Datasets/Dengue_final_dataset_updated.csv')
print("Dataset loaded successfully!!")
print(f"\n Dataset shape: {df.shape}")

Mounted at /content/drive
Dataset loaded successfully!!

 Dataset shape: (23360, 23)


In [ ]:
# Ensure date sorted
df = df.sort_values("Date").reset_index(drop=True)

# Fill missing dengue cases with 0
df["Dengue_Cases"] = df["Dengue_case"].fillna(0)

# ***Create Lagged Dengue Matrix (for IP)***

In [ ]:
MAX_LAG = 21

for k in range(1, MAX_LAG + 1):
    df[f"D_lag_{k}"] = df["Dengue_Cases"].shift(k)

In [10]:
df = df.dropna().reset_index(drop=True)

# ***Learn Optimal Infectious Pressure Weights***

Instead of manually assigning weights, we let the model learn them.

Use LASSO (important for selecting meaningful lags)

In [11]:
from sklearn.linear_model import LassoCV

# Prepare X (lagged dengue)
X_ip = df[[f"D_lag_{k}" for k in range(1, MAX_LAG + 1)]]

# Target
y = df["Dengue_Cases"]

# Fit LASSO
lasso = LassoCV(cv=5, max_iter=10000)
lasso.fit(X_ip, y)

# Extract weights
weights = lasso.coef_

# Display weights
ip_weights = pd.DataFrame({
    "Lag": range(1, MAX_LAG + 1),
    "Weight": weights
})

print(ip_weights)

    Lag    Weight
0     1  0.037289
1     2  0.016788
2     3  0.024378
3     4  0.023471
4     5  0.015803
5     6  0.020567
6     7  0.024542
7     8  0.024521
8     9  0.023524
9    10  0.025018
10   11  0.023002
11   12  0.025371
12   13  0.024777
13   14  0.014884
14   15  0.020737
15   16  0.052432
16   17  0.017211
17   18  0.019783
18   19  0.029721
19   20  0.023425
20   21  0.023062


# **Compute Infectious Pressure**

In [12]:
df["IP"] = 0

for k in range(1, MAX_LAG + 1):
    df["IP"] += weights[k-1] * df[f"D_lag_{k}"]

# Log transform (important)
df["IP_log"] = np.log1p(df["IP"])

# ***Create Rainfall Lag Features***

We use biologically meaningful lags: 7–35 days

In [13]:
RAIN_LAGS = range(7, 36)

for k in RAIN_LAGS:
    df[f"Rain_lag_{k}"] = df["Rainfall"].shift(k)